# Model Development
## 10-Year Coronary Heart Disease Risk Prediction

In this notebook, we developed and compared different machine-learning models to predict the `TenYearCHD` target variable.

The dataset was already cleaned and prepared during the preprocessing stage.

We selected four classification algorithms:

- Logistic Regression
- Decision Tree
- Random Forest
- Support Vector Machine (SVM)

We will evaluate the models using:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix
- ROC-AUC

Since our target variable is imbalanced, we will not depend only on accuracy when comparing the models.

In [0]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)


In [0]:
X_train_scaled = pd.read_csv(
    "/Volumes/workspace/default/fdm_project/X_train_scaled.csv"
)

X_test_scaled = pd.read_csv(
    "/Volumes/workspace/default/fdm_project/X_test_scaled.csv"
)

y_train = pd.read_csv(
    "/Volumes/workspace/default/fdm_project/y_train.csv"
).squeeze()

y_test = pd.read_csv(
    "/Volumes/workspace/default/fdm_project/y_test.csv"
).squeeze()

In [0]:
print("Training features:", X_train_scaled.shape)
print("Testing features:", X_test_scaled.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

### Observation

We loaded the preprocessed training and testing data created during the previous stage.

We checked the shapes of the feature and target datasets to make sure that the data was loaded correctly before starting model development.

## 1. Logistic Regression

We selected Logistic Regression as our first classification model.

Logistic Regression is suitable for binary classification problems because our target variable `TenYearCHD` contains two classes:

- `0` = No CHD within 10 years
- `1` = CHD within 10 years

We used the scaled version of the dataset because Logistic Regression can be affected by differences in feature scales.

This first model is used as a baseline model before applying any hyperparameter tuning or class-imbalance treatment.

In [0]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train_scaled, y_train)

In [0]:
y_pred_lr = logistic_model.predict(X_test_scaled)

y_prob_lr = logistic_model.predict_proba(X_test_scaled)[:, 1]

In [0]:
lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_roc_auc = roc_auc_score(y_test, y_prob_lr)

print("Logistic Regression Results")
print("Accuracy :", round(lr_accuracy, 4))
print("Precision:", round(lr_precision, 4))
print("Recall   :", round(lr_recall, 4))
print("F1-score :", round(lr_f1, 4))
print("ROC-AUC  :", round(lr_roc_auc, 4))

In [0]:
lr_cm = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(5, 4))

sns.heatmap(
    lr_cm,
    annot=True,
    fmt="d"
)

plt.title("Logistic Regression - Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.show()

### Observation

We trained Logistic Regression as our first baseline classification model.

We evaluated the model using accuracy, precision, recall, F1-score, ROC-AUC, and the confusion matrix.

Since the dataset is imbalanced, we will consider all of these measures when comparing the models instead of selecting a model based only on accuracy.

The results of Logistic Regression will later be compared with the other classification algorithms.

### Logistic Regression - Observation

The baseline Logistic Regression model achieved an accuracy of approximately **85.1%**.

However, the recall for the CHD-positive class was only approximately **6.7%**.

From the confusion matrix, the model correctly identified only **22** CHD-positive cases, while **305** actual CHD-positive cases were incorrectly predicted as class `0`.

This shows that accuracy alone is not suitable for evaluating our model because the target variable is imbalanced. The model performs well at identifying the majority class but performs poorly at detecting the minority CHD-positive class.

The precision was approximately **64.7%**, while the F1-score was low at approximately **12.2%** because of the very low recall.

The ROC-AUC value of approximately **0.73** suggests that the model still has some ability to distinguish between the two classes.

Therefore, we will compare this baseline result with other classification algorithms and later investigate suitable methods for handling the class imbalance.

## 2. Decision Tree

We selected Decision Tree as our second classification model.

A Decision Tree makes predictions by splitting the data into different branches based on feature values.

It can capture non-linear relationships between the input features and the target variable.

For this model, we used the unscaled dataset because Decision Tree models do not require feature scaling.

This model will be compared with Logistic Regression and the other classification models.

In [0]:
X_train_clean = pd.read_csv(
    "/Volumes/workspace/default/fdm_project/X_train_clean.csv"
)

X_test_clean = pd.read_csv(
    "/Volumes/workspace/default/fdm_project/X_test_clean.csv"
)

print("Training features:", X_train_clean.shape)
print("Testing features:", X_test_clean.shape)

In [0]:
from sklearn.tree import DecisionTreeClassifier

In [0]:
decision_tree_model = DecisionTreeClassifier(
    random_state=42
)

decision_tree_model.fit(X_train_clean, y_train)

In [0]:
y_pred_dt = decision_tree_model.predict(X_test_clean)

y_prob_dt = decision_tree_model.predict_proba(X_test_clean)[:, 1]

In [0]:
dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_precision = precision_score(y_test, y_pred_dt)
dt_recall = recall_score(y_test, y_pred_dt)
dt_f1 = f1_score(y_test, y_pred_dt)
dt_roc_auc = roc_auc_score(y_test, y_prob_dt)

print("Decision Tree Results")
print("Accuracy :", round(dt_accuracy, 4))
print("Precision:", round(dt_precision, 4))
print("Recall   :", round(dt_recall, 4))
print("F1-score :", round(dt_f1, 4))
print("ROC-AUC  :", round(dt_roc_auc, 4))

In [0]:
dt_cm = confusion_matrix(y_test, y_pred_dt)

plt.figure(figsize=(5, 4))

sns.heatmap(
    dt_cm,
    annot=True,
    fmt="d"
)

plt.title("Decision Tree - Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.show()

### Decision Tree - Observation

The baseline Decision Tree model achieved an accuracy of approximately **87.1%**.

The model achieved a precision of approximately **57.6%** and a recall of approximately **60.2%** for the CHD-positive class.

From the confusion matrix, the model correctly identified **197** CHD-positive cases, while **130** actual CHD-positive cases were incorrectly classified as class `0`.

Compared with the baseline Logistic Regression model, the Decision Tree showed a large improvement in recall and F1-score.

Logistic Regression identified only 22 CHD-positive cases, while the Decision Tree correctly identified 197 cases.

Although the precision of the Decision Tree is slightly lower than Logistic Regression, its recall, F1-score, accuracy, and ROC-AUC are higher.

The ROC-AUC value of approximately **0.76** also indicates better class-separation ability compared with the baseline Logistic Regression model.

At this stage, the Decision Tree provides better overall performance, especially for identifying the minority CHD-positive class. However, the remaining models will also be evaluated before making any model-selection decision.

## 3. Random Forest

We selected Random Forest as our third classification model.

Random Forest combines multiple Decision Trees and uses their predictions together to make the final classification.

This can provide more stable predictions compared with using a single Decision Tree.

For this model, we used the unscaled dataset because Random Forest is a tree-based algorithm and does not require feature scaling.

This baseline model will be compared with Logistic Regression, Decision Tree, and the remaining classification model.

In [0]:
from sklearn.ensemble import RandomForestClassifier

In [0]:
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

random_forest_model.fit(X_train_clean, y_train)

In [0]:
y_pred_rf = random_forest_model.predict(X_test_clean)

y_prob_rf = random_forest_model.predict_proba(X_test_clean)[:, 1]

In [0]:
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)
rf_roc_auc = roc_auc_score(y_test, y_prob_rf)

print("Random Forest Results")
print("Accuracy :", round(rf_accuracy, 4))
print("Precision:", round(rf_precision, 4))
print("Recall   :", round(rf_recall, 4))
print("F1-score :", round(rf_f1, 4))
print("ROC-AUC  :", round(rf_roc_auc, 4))

In [0]:
rf_cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(5, 4))

sns.heatmap(
    rf_cm,
    annot=True,
    fmt="d"
)

plt.title("Random Forest - Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.show()

### Random Forest - Observation

The baseline Random Forest model achieved an accuracy of approximately **92.3%**.

The model achieved a very high precision of approximately **96.6%**, which means that most of the records predicted as CHD-positive were actually positive.

The recall was approximately **51.4%**, meaning that the model correctly detected around half of the actual CHD-positive cases.

From the confusion matrix, the model correctly identified **168** CHD-positive cases, while **159** actual CHD-positive cases were incorrectly predicted as class `0`.

The F1-score was approximately **67.1%**, and the ROC-AUC was approximately **0.95**.

Compared with Logistic Regression and Decision Tree, Random Forest achieved the highest accuracy, precision, F1-score, and ROC-AUC so far.

However, the Decision Tree still achieved a higher recall for the CHD-positive class.

Therefore, we will continue evaluating the remaining model before making a final model-selection decision.

## 4. Support Vector Machine (SVM)

We selected Support Vector Machine as our fourth classification model.

SVM tries to find a decision boundary that separates the two target classes.

It can also capture more complex relationships between features using a kernel.

For this model, we used the scaled dataset because SVM is sensitive to differences in feature scales.

This baseline model will be compared with Logistic Regression, Decision Tree, and Random Forest.

In [0]:
from sklearn.svm import SVC

In [0]:
svm_model = SVC(
    probability=True,
    random_state=42
)

svm_model.fit(X_train_scaled, y_train)

In [0]:
y_pred_svm = svm_model.predict(X_test_scaled)

y_prob_svm = svm_model.predict_proba(X_test_scaled)[:, 1]

In [0]:
svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_precision = precision_score(y_test, y_pred_svm)
svm_recall = recall_score(y_test, y_pred_svm)
svm_f1 = f1_score(y_test, y_pred_svm)
svm_roc_auc = roc_auc_score(y_test, y_prob_svm)

print("SVM Results")
print("Accuracy :", round(svm_accuracy, 4))
print("Precision:", round(svm_precision, 4))
print("Recall   :", round(svm_recall, 4))
print("F1-score :", round(svm_f1, 4))
print("ROC-AUC  :", round(svm_roc_auc, 4))

In [0]:
svm_cm = confusion_matrix(y_test, y_pred_svm)

plt.figure(figsize=(5, 4))

sns.heatmap(
    svm_cm,
    annot=True,
    fmt="d"
)

plt.title("SVM - Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.show()

### SVM - Observation

The baseline SVM model achieved an accuracy of approximately **85.6%**.

The model achieved a very high precision of approximately **95.7%**, meaning that most of the records predicted as CHD-positive were actually positive.

However, the recall was only approximately **6.7%**.

From the confusion matrix, the model correctly identified only **22** CHD-positive cases, while **305** actual CHD-positive cases were incorrectly predicted as class `0`.

This shows that the model strongly favours the majority class and predicts very few records as CHD-positive.

As a result, although the accuracy and precision appear high, the low recall and F1-score show that the baseline SVM is not effective at identifying the minority CHD-positive class.

The ROC-AUC was approximately **0.73**.

Therefore, we will compare the SVM results with the other baseline models and perform further validation and optimization before selecting the final model.

## 5. Baseline Model Comparison

We compared the performance of the four baseline classification models:

- Logistic Regression
- Decision Tree
- Random Forest
- Support Vector Machine (SVM)

Since the target variable is imbalanced, we compared the models using several evaluation measures instead of depending only on accuracy.

The main measures used were accuracy, precision, recall, F1-score, and ROC-AUC.

In [0]:
model_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "SVM"
    ],
    "Accuracy": [
        lr_accuracy,
        dt_accuracy,
        rf_accuracy,
        svm_accuracy
    ],
    "Precision": [
        lr_precision,
        dt_precision,
        rf_precision,
        svm_precision
    ],
    "Recall": [
        lr_recall,
        dt_recall,
        rf_recall,
        svm_recall
    ],
    "F1-score": [
        lr_f1,
        dt_f1,
        rf_f1,
        svm_f1
    ],
    "ROC-AUC": [
        lr_roc_auc,
        dt_roc_auc,
        rf_roc_auc,
        svm_roc_auc
    ]
})

display(model_results.round(4))

### Baseline Model Comparison - Observation

From the baseline comparison, we observed clear differences between the four models.

Random Forest achieved the highest accuracy, precision, F1-score, and ROC-AUC among the four baseline models.

Decision Tree achieved the highest recall and detected a larger proportion of the actual CHD-positive cases.

Logistic Regression and SVM achieved relatively high accuracy, but their recall values were very low. This shows that they predicted the majority class well but failed to identify many CHD-positive cases.

Since our dataset is imbalanced, we cannot select a model using accuracy alone.

Therefore, we considered recall, F1-score, ROC-AUC, and the confusion matrices together when comparing the models.

At the baseline stage, Random Forest showed strong overall performance, while Decision Tree showed the highest recall. Further validation is required before moving to model optimization and final model selection.

## 6. Validation Strategy

We used Stratified 5-Fold Cross-Validation to evaluate whether the model performance is consistent across different parts of the training data.

The training data is divided into five folds.

In each round:

- four folds are used for training
- one fold is used for validation

This process is repeated five times so that every fold is used once for validation.

We used stratification because the target variable is imbalanced. This helps maintain approximately the same class distribution in each fold.

The separate testing dataset is not used during cross-validation.

In [0]:
from sklearn.model_selection import StratifiedKFold, cross_validate

In [0]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [0]:
lr_cv = cross_validate(
    LogisticRegression(max_iter=1000, random_state=42),
    X_train_scaled,
    y_train,
    cv=cv,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"]
)

print("Logistic Regression Cross-Validation")
print("Accuracy :", round(lr_cv["test_accuracy"].mean(), 4))
print("Precision:", round(lr_cv["test_precision"].mean(), 4))
print("Recall   :", round(lr_cv["test_recall"].mean(), 4))
print("F1-score :", round(lr_cv["test_f1"].mean(), 4))
print("ROC-AUC  :", round(lr_cv["test_roc_auc"].mean(), 4))

In [0]:
dt_cv = cross_validate(
    DecisionTreeClassifier(random_state=42),
    X_train_clean,
    y_train,
    cv=cv,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"]
)

print("Decision Tree Cross-Validation")
print("Accuracy :", round(dt_cv["test_accuracy"].mean(), 4))
print("Precision:", round(dt_cv["test_precision"].mean(), 4))
print("Recall   :", round(dt_cv["test_recall"].mean(), 4))
print("F1-score :", round(dt_cv["test_f1"].mean(), 4))
print("ROC-AUC  :", round(dt_cv["test_roc_auc"].mean(), 4))

In [0]:
rf_cv = cross_validate(
    RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    X_train_clean,
    y_train,
    cv=cv,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"]
)

print("Random Forest Cross-Validation")
print("Accuracy :", round(rf_cv["test_accuracy"].mean(), 4))
print("Precision:", round(rf_cv["test_precision"].mean(), 4))
print("Recall   :", round(rf_cv["test_recall"].mean(), 4))
print("F1-score :", round(rf_cv["test_f1"].mean(), 4))
print("ROC-AUC  :", round(rf_cv["test_roc_auc"].mean(), 4))

In [0]:
svm_cv = cross_validate(
    SVC(
        probability=True,
        random_state=42
    ),
    X_train_scaled,
    y_train,
    cv=cv,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"]
)

print("SVM Cross-Validation")
print("Accuracy :", round(svm_cv["test_accuracy"].mean(), 4))
print("Precision:", round(svm_cv["test_precision"].mean(), 4))
print("Recall   :", round(svm_cv["test_recall"].mean(), 4))
print("F1-score :", round(svm_cv["test_f1"].mean(), 4))
print("ROC-AUC  :", round(svm_cv["test_roc_auc"].mean(), 4))

## 7. Cross-Validation Comparison

We compared the average results from Stratified 5-Fold Cross-Validation for all four baseline models.

Using cross-validation allowed us to evaluate each model across different subsets of the training data instead of depending on only one validation split.

Since stratification was used, approximately the same target class distribution was maintained in each fold.

In [0]:
cv_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "SVM"
    ],

    "Accuracy": [
        lr_cv["test_accuracy"].mean(),
        dt_cv["test_accuracy"].mean(),
        rf_cv["test_accuracy"].mean(),
        svm_cv["test_accuracy"].mean()
    ],

    "Precision": [
        lr_cv["test_precision"].mean(),
        dt_cv["test_precision"].mean(),
        rf_cv["test_precision"].mean(),
        svm_cv["test_precision"].mean()
    ],

    "Recall": [
        lr_cv["test_recall"].mean(),
        dt_cv["test_recall"].mean(),
        rf_cv["test_recall"].mean(),
        svm_cv["test_recall"].mean()
    ],

    "F1-score": [
        lr_cv["test_f1"].mean(),
        dt_cv["test_f1"].mean(),
        rf_cv["test_f1"].mean(),
        svm_cv["test_f1"].mean()
    ],

    "ROC-AUC": [
        lr_cv["test_roc_auc"].mean(),
        dt_cv["test_roc_auc"].mean(),
        rf_cv["test_roc_auc"].mean(),
        svm_cv["test_roc_auc"].mean()
    ]
})

display(cv_results.round(4))

### Cross-Validation Observation

The Stratified 5-Fold Cross-Validation results showed a similar pattern to the baseline test-set results.

Random Forest achieved the highest average accuracy, precision, F1-score, and ROC-AUC.

Its average ROC-AUC was approximately **0.92**, indicating strong ability to distinguish between the two target classes.

Decision Tree achieved the highest average recall of approximately **57.0%**, meaning that it detected a larger proportion of the CHD-positive cases compared with the other baseline models.

Logistic Regression and SVM continued to show very low recall values. Although they achieved relatively high accuracy and precision, they failed to identify many of the minority CHD-positive cases.

The cross-validation results therefore support the observations made from the original test-set evaluation.

Random Forest showed the strongest overall performance, while Decision Tree showed the strongest recall. However, further optimization and class-imbalance handling will be investigated before selecting the final model.

# Stage 7 - Model Optimization and Final Model Selection

After comparing the four baseline models, we identified Random Forest and Decision Tree as the stronger candidate models.

Random Forest showed the strongest overall performance, while Decision Tree achieved the highest recall for the CHD-positive class.

Since the target variable is imbalanced, we first investigated whether class-imbalance handling could improve the detection of CHD-positive cases.

After this, we will perform hyperparameter tuning and compare the optimized models with their baseline versions.

## 8.1 Class Imbalance Handling Using Class Weights

Our target variable contains approximately 85% class `0` records and 15% class `1` records.

The baseline models showed that class imbalance affected the ability of some models to identify CHD-positive cases.

We therefore tested class weighting as an imbalance-handling method.

Using `class_weight="balanced"` gives more importance to the minority class during model training.

The testing data was not modified.

In [0]:
weighted_dt = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42
)

weighted_dt.fit(X_train_clean, y_train)

In [0]:
y_pred_dt_weighted = weighted_dt.predict(X_test_clean)

y_prob_dt_weighted = weighted_dt.predict_proba(X_test_clean)[:, 1]

In [0]:
weighted_dt_cm = confusion_matrix(
    y_test,
    y_pred_dt_weighted
)

plt.figure(figsize=(5, 4))

sns.heatmap(
    weighted_dt_cm,
    annot=True,
    fmt="d"
)

plt.title("Weighted Decision Tree - Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.show()

### Weighted Decision Tree - Observation

After applying `class_weight="balanced"`, the Decision Tree showed an improvement compared with the original baseline model.

The weighted model correctly identified **200** CHD-positive cases, compared with **197** in the original Decision Tree.

The number of false negatives decreased from **130 to 127**, meaning that slightly fewer CHD-positive cases were missed.

The weighted model also reduced false positives from **145 to 115**.

The recall increased from approximately **60.2% to 61.2%**, while the precision and F1-score also improved.

These results suggest that applying class weighting improved the Decision Tree's overall performance while slightly improving its ability to identify the minority CHD-positive class.

## 8.2 Weighted Random Forest

Random Forest showed the strongest overall performance among the baseline models.

However, its recall for the CHD-positive class was lower than the Decision Tree.

Therefore, we tested `class_weight="balanced"` to investigate whether giving more importance to the minority class improves the detection of CHD-positive cases.

All other main settings were kept the same as the baseline Random Forest so that we could compare the effect of class weighting.

In [0]:
weighted_rf = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)

weighted_rf.fit(X_train_clean, y_train)


In [0]:
y_pred_rf_weighted = weighted_rf.predict(X_test_clean)

y_prob_rf_weighted = weighted_rf.predict_proba(X_test_clean)[:, 1]

In [0]:
weighted_rf_accuracy = accuracy_score(y_test, y_pred_rf_weighted)
weighted_rf_precision = precision_score(y_test, y_pred_rf_weighted)
weighted_rf_recall = recall_score(y_test, y_pred_rf_weighted)
weighted_rf_f1 = f1_score(y_test, y_pred_rf_weighted)
weighted_rf_auc = roc_auc_score(y_test, y_prob_rf_weighted)

print("Weighted Random Forest Results")
print("Accuracy :", round(weighted_rf_accuracy, 4))
print("Precision:", round(weighted_rf_precision, 4))
print("Recall   :", round(weighted_rf_recall, 4))
print("F1-score :", round(weighted_rf_f1, 4))
print("ROC-AUC  :", round(weighted_rf_auc, 4))

In [0]:
weighted_rf_cm = confusion_matrix(
    y_test,
    y_pred_rf_weighted
)

plt.figure(figsize=(5, 4))

sns.heatmap(
    weighted_rf_cm,
    annot=True,
    fmt="d"
)

plt.title("Weighted Random Forest - Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.show()

### Weighted Random Forest - Observation

After applying `class_weight="balanced"`, the Random Forest showed only small changes compared with the baseline model.

The precision increased slightly from approximately **96.6% to 97.6%**, and the ROC-AUC also increased slightly.

However, the recall decreased from approximately **51.4% to 49.9%**.

The number of correctly identified CHD-positive cases decreased from **168 to 163**, while false negatives increased from **159 to 164**.

The F1-score and accuracy also decreased slightly.

Therefore, class weighting did not improve the Random Forest's ability to identify the minority CHD-positive class.

For Random Forest, the original baseline model performed better overall than the weighted version.

## 8.3 Comparison of Class-Weight Experiments

We compared the baseline and class-weighted versions of Decision Tree and Random Forest.

The purpose was to determine whether giving more importance to the minority CHD-positive class improved model performance.

We compared accuracy, precision, recall, F1-score, and ROC-AUC.

In [0]:
weighted_dt_accuracy = accuracy_score(y_test, y_pred_dt_weighted)
weighted_dt_precision = precision_score(y_test, y_pred_dt_weighted)
weighted_dt_recall = recall_score(y_test, y_pred_dt_weighted)
weighted_dt_f1 = f1_score(y_test, y_pred_dt_weighted)
weighted_dt_auc = roc_auc_score(y_test, y_prob_dt_weighted)

print("Weighted Decision Tree Results")
print("Accuracy :", round(weighted_dt_accuracy, 4))
print("Precision:", round(weighted_dt_precision, 4))
print("Recall   :", round(weighted_dt_recall, 4))
print("F1-score :", round(weighted_dt_f1, 4))
print("ROC-AUC  :", round(weighted_dt_auc, 4))

In [0]:
weight_comparison = pd.DataFrame({
    "Model": [
        "Baseline Decision Tree",
        "Weighted Decision Tree",
        "Baseline Random Forest",
        "Weighted Random Forest"
    ],

    "Accuracy": [
        dt_accuracy,
        weighted_dt_accuracy,
        rf_accuracy,
        weighted_rf_accuracy
    ],

    "Precision": [
        dt_precision,
        weighted_dt_precision,
        rf_precision,
        weighted_rf_precision
    ],

    "Recall": [
        dt_recall,
        weighted_dt_recall,
        rf_recall,
        weighted_rf_recall
    ],

    "F1-score": [
        dt_f1,
        weighted_dt_f1,
        rf_f1,
        weighted_rf_f1
    ],

    "ROC-AUC": [
        dt_roc_auc,
        weighted_dt_auc,
        rf_roc_auc,
        weighted_rf_auc
    ]
})

display(weight_comparison.round(4))

### Observation

The class-weight experiment affected the two models differently.

For Decision Tree, using `class_weight="balanced"` improved accuracy, precision, recall, and F1-score. It also slightly reduced the number of CHD-positive cases that were missed.

For Random Forest, class weighting slightly improved precision and ROC-AUC, but recall and F1-score decreased.

Therefore, class weighting was useful for Decision Tree but did not improve the overall performance of Random Forest.

Based on these results, we kept the weighted Decision Tree and the baseline Random Forest as the stronger candidates for further optimization.

## 9. Hyperparameter Tuning

After identifying the stronger candidate models, we performed hyperparameter tuning.

Hyperparameters are settings that control how a machine-learning model is built.

We used `GridSearchCV` to test different combinations of hyperparameter values.

Stratified 5-Fold Cross-Validation was used during tuning so that the class distribution remained approximately the same in each fold.

The tuning process used only the training data. The test data was kept separate for final evaluation.

In [0]:
from sklearn.model_selection import GridSearchCV

In [0]:
dt_param_grid = {
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5]
}

In [0]:
dt_grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),
    param_grid=dt_param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1
)

dt_grid_search.fit(X_train_clean, y_train)

In [0]:
print("Best Decision Tree Parameters:")
print(dt_grid_search.best_params_)

print("\nBest Cross-Validation F1-score:")
print(round(dt_grid_search.best_score_, 4))

### Decision Tree Hyperparameter Tuning - Observation

GridSearchCV selected the following hyperparameters:

- `max_depth = None`
- `min_samples_leaf = 1`
- `min_samples_split = 2`

The best average cross-validation F1-score was approximately **0.535**.

These values are the same as the default tree-complexity settings used in the weighted Decision Tree.

This indicates that, within the parameter combinations tested, limiting the tree depth or increasing the minimum number of samples did not improve the cross-validation F1-score.

Therefore, the tuning process did not identify a more suitable Decision Tree structure than the weighted model already tested.

In [0]:
best_dt_model = dt_grid_search.best_estimator_

y_pred_dt_tuned = best_dt_model.predict(X_test_clean)
y_prob_dt_tuned = best_dt_model.predict_proba(X_test_clean)[:, 1]

In [0]:
tuned_dt_accuracy = accuracy_score(y_test, y_pred_dt_tuned)
tuned_dt_precision = precision_score(y_test, y_pred_dt_tuned)
tuned_dt_recall = recall_score(y_test, y_pred_dt_tuned)
tuned_dt_f1 = f1_score(y_test, y_pred_dt_tuned)
tuned_dt_auc = roc_auc_score(y_test, y_prob_dt_tuned)

print("Tuned Decision Tree Results")
print("Accuracy :", round(tuned_dt_accuracy, 4))
print("Precision:", round(tuned_dt_precision, 4))
print("Recall   :", round(tuned_dt_recall, 4))
print("F1-score :", round(tuned_dt_f1, 4))
print("ROC-AUC  :", round(tuned_dt_auc, 4))

In [0]:
tuned_dt_cm = confusion_matrix(y_test, y_pred_dt_tuned)

plt.figure(figsize=(5, 4))

sns.heatmap(
    tuned_dt_cm,
    annot=True,
    fmt="d"
)

plt.title("Tuned Decision Tree - Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.show()

## 9.2 Random Forest Hyperparameter Tuning

Random Forest showed the strongest overall performance among the baseline models.

The class-weighted Random Forest did not improve recall or F1-score, so we continued tuning the original Random Forest model.

We used GridSearchCV with Stratified 5-Fold Cross-Validation to test different combinations of Random Forest hyperparameters.

The tuning process was performed only using the training data.

In [0]:
rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

In [0]:
rf_grid_search = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=42
    ),
    param_grid=rf_param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1
)

rf_grid_search.fit(X_train_clean, y_train)

In [0]:
print("Best Random Forest Parameters:")
print(rf_grid_search.best_params_)

print("\nBest Cross-Validation F1-score:")
print(round(rf_grid_search.best_score_, 4))

In [0]:
best_rf_model = rf_grid_search.best_estimator_

In [0]:
y_pred_rf_tuned = best_rf_model.predict(X_test_clean)

y_prob_rf_tuned = best_rf_model.predict_proba(
    X_test_clean
)[:, 1]

In [0]:
tuned_rf_accuracy = accuracy_score(y_test, y_pred_rf_tuned)
tuned_rf_precision = precision_score(y_test, y_pred_rf_tuned)
tuned_rf_recall = recall_score(y_test, y_pred_rf_tuned)
tuned_rf_f1 = f1_score(y_test, y_pred_rf_tuned)
tuned_rf_auc = roc_auc_score(y_test, y_prob_rf_tuned)

print("Tuned Random Forest Results")
print("Accuracy :", round(tuned_rf_accuracy, 4))
print("Precision:", round(tuned_rf_precision, 4))
print("Recall   :", round(tuned_rf_recall, 4))
print("F1-score :", round(tuned_rf_f1, 4))
print("ROC-AUC  :", round(tuned_rf_auc, 4))

In [0]:
tuned_rf_cm = confusion_matrix(
    y_test,
    y_pred_rf_tuned
)

plt.figure(figsize=(5, 4))

sns.heatmap(
    tuned_rf_cm,
    annot=True,
    fmt="d"
)

plt.title("Tuned Random Forest - Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.show()

### Random Forest Hyperparameter Tuning - Observation

GridSearchCV selected the following Random Forest hyperparameters:

- `n_estimators = 200`
- `max_depth = None`
- `min_samples_split = 2`
- `min_samples_leaf = 1`

The best cross-validation F1-score was approximately **0.619**.

Compared with the baseline Random Forest, the tuned model improved across all main evaluation measures.

The accuracy increased from approximately **92.3% to 92.8%**, while precision increased from **96.6% to 97.3%**.

Recall improved from approximately **51.4% to 54.7%**, meaning that the tuned model identified more of the actual CHD-positive cases.

The number of correctly identified CHD-positive cases increased from **168 to 179**, while false negatives decreased from **159 to 148**.

The F1-score increased from approximately **0.671 to 0.701**, and ROC-AUC also increased slightly from approximately **0.954 to 0.957**.

Therefore, hyperparameter tuning improved the overall performance of the Random Forest model.

## 10. Final Model Comparison

After completing the baseline evaluation, class-weight experiments, and hyperparameter tuning, we compared the stronger Decision Tree and Random Forest models.

The purpose of this comparison was to determine which model provides the most suitable overall performance for our project.

We considered accuracy, precision, recall, F1-score, and ROC-AUC instead of selecting a model using accuracy alone.

In [0]:
final_comparison = pd.DataFrame({
    "Model": [
        "Baseline Decision Tree",
        "Weighted Decision Tree",
        "Tuned Decision Tree",
        "Baseline Random Forest",
        "Weighted Random Forest",
        "Tuned Random Forest"
    ],

    "Accuracy": [
        dt_accuracy,
        weighted_dt_accuracy,
        tuned_dt_accuracy,
        rf_accuracy,
        weighted_rf_accuracy,
        tuned_rf_accuracy
    ],

    "Precision": [
        dt_precision,
        weighted_dt_precision,
        tuned_dt_precision,
        rf_precision,
        weighted_rf_precision,
        tuned_rf_precision
    ],

    "Recall": [
        dt_recall,
        weighted_dt_recall,
        tuned_dt_recall,
        rf_recall,
        weighted_rf_recall,
        tuned_rf_recall
    ],

    "F1-score": [
        dt_f1,
        weighted_dt_f1,
        tuned_dt_f1,
        rf_f1,
        weighted_rf_f1,
        tuned_rf_f1
    ],

    "ROC-AUC": [
        dt_roc_auc,
        weighted_dt_auc,
        tuned_dt_auc,
        rf_roc_auc,
        weighted_rf_auc,
        tuned_rf_auc
    ]
})

display(final_comparison.round(4))

## 11. Final Model Selection

After comparing the baseline, class-weighted, and tuned models, the Tuned Random Forest showed the strongest overall performance.

The Tuned Random Forest achieved:

- Accuracy of approximately **92.8%**
- Precision of approximately **97.3%**
- Recall of approximately **54.7%**
- F1-score of approximately **70.1%**
- ROC-AUC of approximately **95.7%**

The tuned model correctly identified **179** CHD-positive cases and reduced the number of false negatives to **148**.

Decision Tree achieved a higher recall, meaning that it detected a larger proportion of the CHD-positive class. However, Random Forest achieved a stronger balance across precision, recall, F1-score, accuracy, and ROC-AUC.

Therefore, based on the overall evaluation results, we selected the Tuned Random Forest as the final model for this project.

## 12. Effect of the Engineered `pulsePressure` Feature

During preprocessing, we created the `pulsePressure` feature using:

`pulsePressure = sysBP - diaBP`

To check whether this engineered feature improves model performance, we compared the Tuned Random Forest with and without `pulsePressure`.

The same Random Forest settings were used for both models so that the effect of the feature could be compared fairly.

In [0]:
X_train_no_pulse = X_train_clean.drop(columns=["pulsePressure"])
X_test_no_pulse = X_test_clean.drop(columns=["pulsePressure"])

print("With pulsePressure:", X_train_clean.shape)
print("Without pulsePressure:", X_train_no_pulse.shape)

In [0]:
rf_no_pulse = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42
)

rf_no_pulse.fit(X_train_no_pulse, y_train)

In [0]:
y_pred_no_pulse = rf_no_pulse.predict(X_test_no_pulse)

y_prob_no_pulse = rf_no_pulse.predict_proba(
    X_test_no_pulse
)[:, 1]

In [0]:
no_pulse_accuracy = accuracy_score(y_test, y_pred_no_pulse)
no_pulse_precision = precision_score(y_test, y_pred_no_pulse)
no_pulse_recall = recall_score(y_test, y_pred_no_pulse)
no_pulse_f1 = f1_score(y_test, y_pred_no_pulse)
no_pulse_auc = roc_auc_score(y_test, y_prob_no_pulse)

print("Random Forest Without pulsePressure")
print("Accuracy :", round(no_pulse_accuracy, 4))
print("Precision:", round(no_pulse_precision, 4))
print("Recall   :", round(no_pulse_recall, 4))
print("F1-score :", round(no_pulse_f1, 4))
print("ROC-AUC  :", round(no_pulse_auc, 4))

In [0]:
pulse_comparison = pd.DataFrame({
    "Model": [
        "Tuned RF with pulsePressure",
        "Tuned RF without pulsePressure"
    ],

    "Accuracy": [
        tuned_rf_accuracy,
        no_pulse_accuracy
    ],

    "Precision": [
        tuned_rf_precision,
        no_pulse_precision
    ],

    "Recall": [
        tuned_rf_recall,
        no_pulse_recall
    ],

    "F1-score": [
        tuned_rf_f1,
        no_pulse_f1
    ],

    "ROC-AUC": [
        tuned_rf_auc,
        no_pulse_auc
    ]
})

display(pulse_comparison.round(4))

### Feature Engineering Experiment - Observation

We compared the Tuned Random Forest with and without the engineered `pulsePressure` feature.

After removing `pulsePressure`, the model showed a small improvement across all evaluation measures.

- Accuracy increased from approximately **92.8% to 93.0%**
- Precision increased from approximately **97.3% to 98.4%**
- Recall increased from approximately **54.7% to 55.4%**
- F1-score increased from approximately **70.1% to 70.8%**
- ROC-AUC increased from approximately **95.7% to 96.2%**

This suggests that the engineered `pulsePressure` feature did not provide additional useful information to the Random Forest model.

Since `pulsePressure` is calculated directly from `sysBP` and `diaBP`, it may contain redundant information.

Therefore, we decided not to include `pulsePressure` in the final feature set.

In [0]:
rf_with_pulse_cv = cross_validate(
    RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42
    ),
    X_train_clean,
    y_train,
    cv=cv,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"]
)

rf_without_pulse_cv = cross_validate(
    RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42
    ),
    X_train_no_pulse,
    y_train,
    cv=cv,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"]
)

In [0]:
feature_cv_comparison = pd.DataFrame({
    "Model": [
        "With pulsePressure",
        "Without pulsePressure"
    ],

    "Accuracy": [
        rf_with_pulse_cv["test_accuracy"].mean(),
        rf_without_pulse_cv["test_accuracy"].mean()
    ],

    "Precision": [
        rf_with_pulse_cv["test_precision"].mean(),
        rf_without_pulse_cv["test_precision"].mean()
    ],

    "Recall": [
        rf_with_pulse_cv["test_recall"].mean(),
        rf_without_pulse_cv["test_recall"].mean()
    ],

    "F1-score": [
        rf_with_pulse_cv["test_f1"].mean(),
        rf_without_pulse_cv["test_f1"].mean()
    ],

    "ROC-AUC": [
        rf_with_pulse_cv["test_roc_auc"].mean(),
        rf_without_pulse_cv["test_roc_auc"].mean()
    ]
})

display(feature_cv_comparison.round(4))

In [0]:
display(feature_cv_comparison.round(4))

### Final Feature Engineering Decision

We compared the Tuned Random Forest with and without the engineered `pulsePressure` feature using Stratified 5-Fold Cross-Validation.

The model without `pulsePressure` achieved slightly higher average accuracy, precision, recall, F1-score, and ROC-AUC.

The average F1-score increased from approximately **0.6190 to 0.6228**, while ROC-AUC increased from approximately **0.9296 to 0.9333**.

Since `pulsePressure` is calculated directly from `sysBP` and `diaBP`, it may provide redundant information rather than additional useful information.

Based on the cross-validation results, we decided to remove `pulsePressure` from the final feature set.

In [0]:
rf_final_grid_search = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=42
    ),
    param_grid=rf_param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1
)

rf_final_grid_search.fit(
    X_train_no_pulse,
    y_train
)

In [0]:
print("Best Final Random Forest Parameters:")
print(rf_final_grid_search.best_params_)

print("\nBest Cross-Validation F1-score:")
print(round(rf_final_grid_search.best_score_, 4))

### Final Random Forest Tuning - Observation

After removing the `pulsePressure` feature, we repeated the Random Forest hyperparameter tuning using Stratified 5-Fold Cross-Validation.

The best parameters were:

- `n_estimators = 200`
- `max_depth = None`
- `min_samples_split = 2`
- `min_samples_leaf = 1`

The best average cross-validation F1-score was approximately **0.6228**.

The selected parameters were the same as the previous tuned Random Forest, confirming that these settings remained suitable after removing the `pulsePressure` feature.

In [0]:
final_model = rf_final_grid_search.best_estimator_

In [0]:
y_pred_final = final_model.predict(X_test_no_pulse)

y_prob_final = final_model.predict_proba(
    X_test_no_pulse
)[:, 1]

In [0]:
final_accuracy = accuracy_score(y_test, y_pred_final)
final_precision = precision_score(y_test, y_pred_final)
final_recall = recall_score(y_test, y_pred_final)
final_f1 = f1_score(y_test, y_pred_final)
final_auc = roc_auc_score(y_test, y_prob_final)

print("Final Random Forest Results")
print("Accuracy :", round(final_accuracy, 4))
print("Precision:", round(final_precision, 4))
print("Recall   :", round(final_recall, 4))
print("F1-score :", round(final_f1, 4))
print("ROC-AUC  :", round(final_auc, 4))

In [0]:
final_cm = confusion_matrix(y_test, y_pred_final)

plt.figure(figsize=(5, 4))

sns.heatmap(
    final_cm,
    annot=True,
    fmt="d"
)

plt.title("Final Random Forest - Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.show()

## 13. Final Model Evaluation

After completing hyperparameter tuning and feature-selection experiments, we evaluated the final Random Forest model.

The final model used the following hyperparameters:

- `n_estimators = 200`
- `max_depth = None`
- `min_samples_split = 2`
- `min_samples_leaf = 1`

The engineered `pulsePressure` feature was removed because the cross-validation experiment showed slightly better performance without it.

### Final Results

The final Random Forest achieved:

- Accuracy: **93.0%**
- Precision: **98.4%**
- Recall: **55.4%**
- F1-score: **70.8%**
- ROC-AUC: **96.2%**

From the confusion matrix, the final model correctly identified **181** CHD-positive cases, while **146** CHD-positive cases were incorrectly predicted as class `0`.

Compared with the baseline Random Forest, the final model improved across all main evaluation measures.

The number of correctly identified CHD-positive cases increased from **168 to 181**, while false negatives decreased from **159 to 146**.

Therefore, the final tuned Random Forest without the `pulsePressure` feature was selected as the final model for the 10-Year CHD Risk Prediction System.

In [0]:
final_features = X_train_no_pulse.columns.tolist()

print("Number of final features:", len(final_features))
print(final_features)

In [0]:
import joblib

model_path = "/Volumes/workspace/default/fdm_project/final_random_forest_model.pkl"

joblib.dump(final_model, model_path)

print("Final model saved successfully.")
print("Path:", model_path)

In [0]:
feature_path = "/Volumes/workspace/default/fdm_project/final_feature_names.pkl"

joblib.dump(final_features, feature_path)

print("Feature names saved successfully.")

In [0]:
loaded_model = joblib.load(
    "/Volumes/workspace/default/fdm_project/final_random_forest_model.pkl"
)

loaded_features = joblib.load(
    "/Volumes/workspace/default/fdm_project/final_feature_names.pkl"
)

print("Model loaded successfully.")
print("Features:", loaded_features)

In [0]:
sample = X_test_no_pulse.iloc[[0]]

prediction = loaded_model.predict(sample)
probability = loaded_model.predict_proba(sample)[:, 1]

print("Prediction:", prediction[0])
print("Class 1 probability:", probability[0])

In [0]:
import pandas as pd

def preprocess_user_input(data):

    processed = {
        "gender": 1 if data["gender"] == "Male" else 0,
        "age": float(data["age"]),
        "education": int(data["education"]),
        "currentSmoker": 1 if data["currentSmoker"] == "Yes" else 0,
        "cigsPerDay": float(data["cigsPerDay"]),
        "BPMeds": 1 if data["BPMeds"] == "Yes" else 0,
        "prevalentStroke": 1 if data["prevalentStroke"] == "Yes" else 0,
        "prevalentHyp": 1 if data["prevalentHyp"] == "Yes" else 0,
        "diabetes": 1 if data["diabetes"] == "Yes" else 0,
        "totChol": float(data["totChol"]),
        "sysBP": float(data["sysBP"]),
        "diaBP": float(data["diaBP"]),
        "BMI": float(data["BMI"]),
        "heartRate": float(data["heartRate"]),
        "glucose": float(data["glucose"])
    }

    input_df = pd.DataFrame(
        [processed],
        columns=loaded_features
    )

    return input_df

In [0]:
example_user = {
    "gender": "Male",
    "age": 55,
    "education": 2,
    "currentSmoker": "Yes",
    "cigsPerDay": 10,
    "BPMeds": "No",
    "prevalentStroke": "No",
    "prevalentHyp": "Yes",
    "diabetes": "No",
    "totChol": 230,
    "sysBP": 145,
    "diaBP": 90,
    "BMI": 27.5,
    "heartRate": 75,
    "glucose": 90
}

processed_user = preprocess_user_input(example_user)

display(processed_user)

In [0]:
user_prediction = loaded_model.predict(processed_user)[0]

user_probability = loaded_model.predict_proba(
    processed_user
)[0, 1]

print("Prediction:", user_prediction)
print("Class 1 probability:", round(user_probability, 4))

In [0]:
import sklearn
import joblib
import pandas

print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("pandas:", pandas.__version__)